# Step 3 — SQL Database Setup & Advanced Querying
## Project: Women's Safety in India — Critical Risk Zone Analysis
### Notebook: 02_SQL_Analysis.ipynb


In [1]:
# =============================================================
# STEP 3.1 — SQLite Database Setup & Data Loading
# =============================================================

import pandas as pd
import sqlite3
import os

# --- Path Configuration ---
BASE_DIR = r"C:\Users\diyas\OneDrive\Desktop\women_safety_india"

CSV_PATH = os.path.join(BASE_DIR, "data", "cleaned", "master_crime_data_clean.csv")
DB_PATH  = os.path.join(BASE_DIR, "data", "cleaned", "women_safety.db")

# --- Safety Check ---
if not os.path.exists(CSV_PATH):
    print(f"❌ CSV not found at: {CSV_PATH}")
else:
    print("✅ CSV found — proceeding...")

print("=" * 60)
print("STEP 3.1 — SQLite Database Setup & Data Loading")
print("=" * 60)

# ── Load CSV ──────────────────────────────────────────────────
print("\n[1/5] Loading cleaned CSV...")
df = pd.read_csv(CSV_PATH)
print(f"      ✅ CSV Loaded Successfully")
print(f"      Shape   : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"      Columns : {list(df.columns)}")

# ── Enforce Data Types ────────────────────────────────────────
print("\n[2/5] Enforcing data types...")

dtype_map = {
    'Year'                    : 'int64',
    'Pop_2024_Est'            : 'int64',
    'Total_Violence'          : 'int64',
    'Crime_Rate'              : 'float64',
    'Police_Vacancy_Rate'     : 'float64',
    'Spousal_Violence_Percent': 'float64',
    'Sought_Help_Percent'     : 'float64',
    'State'                   : 'object',
    'District'                : 'object'
}

for col, dtype in dtype_map.items():
    if col in df.columns:
        df[col] = df[col].astype(dtype)
    else:
        print(f"      ⚠️ WARNING: '{col}' not found in CSV")

print("      ✅ Data types enforced")
print(df.dtypes.to_string())

# ── Write to SQLite ───────────────────────────────────────────
print("\n[3/5] Writing to SQLite database...")

conn = sqlite3.connect(DB_PATH)

df.to_sql(
    name      = 'crime_data',
    con       = conn,
    if_exists = 'replace',
    index     = False
)

print(f"      ✅ Table 'crime_data' written successfully")
print(f"      📁 Database saved at: {DB_PATH}")

# ── Verification Queries ──────────────────────────────────────
print("\n[4/5] Running verification queries...")

# Row Count
q1      = "SELECT COUNT(*) AS total_rows FROM crime_data;"
db_rows = pd.read_sql_query(q1, conn)['total_rows'].iloc[0]
print(f"\n      📊 Row Count:")
print(f"         CSV : {df.shape[0]} rows")
print(f"         DB  : {db_rows} rows")
print(f"         {'✅ MATCH' if db_rows == df.shape[0] else '❌ MISMATCH'}")

# Year Range
q2 = """
    SELECT
        MIN(Year)            AS earliest_year,
        MAX(Year)            AS latest_year,
        COUNT(DISTINCT Year) AS unique_years
    FROM crime_data;
"""
print(f"\n      📊 Year Range:")
display(pd.read_sql_query(q2, conn))

# State & District Count
q3 = """
    SELECT
        COUNT(DISTINCT State)    AS total_states,
        COUNT(DISTINCT District) AS total_districts
    FROM crime_data;
"""
print(f"\n      📊 States & Districts:")
display(pd.read_sql_query(q3, conn))

# NULL Check
q4 = """
    SELECT
        SUM(CASE WHEN Year                     IS NULL THEN 1 ELSE 0 END) AS null_Year,
        SUM(CASE WHEN State                    IS NULL THEN 1 ELSE 0 END) AS null_State,
        SUM(CASE WHEN District                 IS NULL THEN 1 ELSE 0 END) AS null_District,
        SUM(CASE WHEN Crime_Rate               IS NULL THEN 1 ELSE 0 END) AS null_Crime_Rate,
        SUM(CASE WHEN Police_Vacancy_Rate      IS NULL THEN 1 ELSE 0 END) AS null_Police_Vacancy,
        SUM(CASE WHEN Spousal_Violence_Percent IS NULL THEN 1 ELSE 0 END) AS null_Spousal,
        SUM(CASE WHEN Sought_Help_Percent      IS NULL THEN 1 ELSE 0 END) AS null_Sought_Help
    FROM crime_data;
"""
print(f"\n      📊 NULL Check:")
display(pd.read_sql_query(q4, conn))

# Preview
print(f"\n      📊 First 5 Rows from Database:")
display(pd.read_sql_query("SELECT * FROM crime_data LIMIT 5;", conn))

# ── Schema Inspection ─────────────────────────────────────────
print("\n[5/5] Table Schema:")
display(pd.read_sql_query("PRAGMA table_info(crime_data);", conn))

# ── Close Connection ──────────────────────────────────────────
conn.close()

print("\n" + "=" * 60)
print("✅ STEP 3.1 COMPLETE — Database ready for querying")
print("=" * 60)

✅ CSV found — proceeding...
STEP 3.1 — SQLite Database Setup & Data Loading

[1/5] Loading cleaned CSV...
      ✅ CSV Loaded Successfully
      Shape   : 3976 rows × 9 columns
      Columns : ['Year', 'State', 'District', 'Pop_2024_Est', 'Total_Violence', 'Crime_Rate', 'Police_Vacancy_Rate', 'Spousal_Violence_Percent', 'Sought_Help_Percent']

[2/5] Enforcing data types...
      ✅ Data types enforced
Year                          int64
State                        object
District                     object
Pop_2024_Est                  int64
Total_Violence                int64
Crime_Rate                  float64
Police_Vacancy_Rate         float64
Spousal_Violence_Percent    float64
Sought_Help_Percent         float64

[3/5] Writing to SQLite database...
      ✅ Table 'crime_data' written successfully
      📁 Database saved at: C:\Users\diyas\OneDrive\Desktop\women_safety_india\data\cleaned\women_safety.db

[4/5] Running verification queries...

      📊 Row Count:
         CSV : 3976 ro

,earliest_year,latest_year,unique_years
0,2021,2024,4



      📊 States & Districts:


,total_states,total_districts
0,37,986



      📊 NULL Check:


...


      📊 First 5 Rows from Database:


,Year,State,District,Pop_2024_Est,Total_Violence,Crime_Rate,Police_Vacancy_Rate,Spousal_Violence_Percent,Sought_Help_Percent
0,2021,Andhra Pradesh,Anantapur,4693320,405,8.629286,0.171097,30.0,4.4
1,2021,Andhra Pradesh,Chittoor,4800173,352,7.333069,0.171097,30.0,4.4
2,2021,Andhra Pradesh,Cuddapah,4427452,341,7.701947,0.171097,30.0,4.4
3,2021,Andhra Pradesh,East Godavari,5927440,523,8.823371,0.171097,30.0,4.4
4,2021,Andhra Pradesh,Guntakal Railway,4427452,3,0.067759,0.171097,30.0,4.4



[5/5] Table Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,Year,INTEGER,0,None,0
1,1,State,TEXT,0,None,0
2,2,District,TEXT,0,None,0
3,3,Pop_2024_Est,INTEGER,0,None,0
4,4,Total_Violence,INTEGER,0,None,0
5,5,Crime_Rate,REAL,0,None,0
6,6,Police_Vacancy_Rate,REAL,0,None,0
7,7,Spousal_Violence_Percent,REAL,0,None,0
8,8,Sought_Help_Percent,REAL,0,None,0



✅ STEP 3.1 COMPLETE — Database ready for querying


## Step 3.2 — Exploratory SQL Queries
### Goal: Understand data distribution before building Red Zone logic


# =============================================================
# STEP 3.2 — Exploratory SQL Queries
# =============================================================

import pandas as pd
import sqlite3
import os

# --- Reopen Connection ---
BASE_DIR = r"C:\Users\diyas\OneDrive\Desktop\women_safety_india"
DB_PATH  = os.path.join(BASE_DIR, "data", "cleaned", "women_safety.db")

conn = sqlite3.connect(DB_PATH)
print("✅ Connected to women_safety.db")
print("=" * 60)

# =============================================================
# QUERY 1 — Top 10 Most Dangerous Districts by Crime Rate
# =============================================================
# WHY: Establishes the absolute worst performers.
#      This becomes the baseline for Red Zone comparison.

print("\n📊 QUERY 1: Top 10 Most Dangerous Districts")
print("-" * 60)

q1 = """
    SELECT
        State,
        District,
        Year,
        Crime_Rate,
        Total_Violence,
        Pop_2024_Est
    FROM crime_data
    ORDER BY Crime_Rate DESC
    LIMIT 10;
"""

result1 = pd.read_sql_query(q1, conn)
display(result1)

# =============================================================
# QUERY 2 — State Level Aggregated Crime Summary
# =============================================================
# WHY: Identifies which STATES are systemic hotspots vs
#      which have isolated district-level problems.
#      Critical for policy-level recommendations.

print("\n📊 QUERY 2: State-Level Crime Summary (Ranked by Avg Crime Rate)")
print("-" * 60)

q2 = """
    SELECT
        State,
        COUNT(DISTINCT District)          AS total_districts,
        ROUND(AVG(Crime_Rate), 2)         AS avg_crime_rate,
        ROUND(MAX(Crime_Rate), 2)         AS max_crime_rate,
        ROUND(AVG(Police_Vacancy_Rate),2) AS avg_police_vacancy,
        ROUND(AVG(Sought_Help_Percent),2) AS avg_help_seeking
    FROM crime_data
    GROUP BY State
    ORDER BY avg_crime_rate DESC
    LIMIT 15;
"""

result2 = pd.read_sql_query(q2, conn)
display(result2)

# =============================================================
# QUERY 3 — Police Vacancy Distribution
# =============================================================
# WHY: Categorizes districts into vacancy severity buckets.
#      Reveals systemic resource misallocation patterns
#      across the entire country at a glance.

print("\n📊 QUERY 3: Police Vacancy Severity Distribution")
print("-" * 60)

q3 = """
    SELECT
        CASE
            WHEN Police_Vacancy_Rate >= 40 THEN 'CRITICAL  (>=40%)'
            WHEN Police_Vacancy_Rate >= 25 THEN 'HIGH      (25-39%)'
            WHEN Police_Vacancy_Rate >= 10 THEN 'MODERATE  (10-24%)'
            ELSE                                'LOW       (<10%)'
        END                        AS vacancy_severity,
        COUNT(*)                   AS district_count,
        ROUND(AVG(Crime_Rate), 2)  AS avg_crime_rate
    FROM crime_data
    GROUP BY vacancy_severity
    ORDER BY district_count DESC;
"""

result3 = pd.read_sql_query(q3, conn)
display(result3)

# =============================================================
# QUERY 4 — Underreporting & Justice Gap Analysis
# =============================================================
# WHY: The CORE of our project thesis.
#      High Spousal_Violence + Low Sought_Help = 
#      Silent suffering zones where crimes go unreported.
#      These are JUSTICE FAILURE zones, not just crime zones.

print("\n📊 QUERY 4: Underreporting & Justice Gap Analysis")
print("-" * 60)

q4 = """
    SELECT
        State,
        District,
        ROUND(Spousal_Violence_Percent, 2) AS spousal_violence_pct,
        ROUND(Sought_Help_Percent, 2)      AS sought_help_pct,
        ROUND(
            Spousal_Violence_Percent - Sought_Help_Percent
        , 2)                               AS justice_gap,
        ROUND(Crime_Rate, 2)               AS crime_rate
    FROM crime_data
    WHERE Spousal_Violence_Percent > 30
    ORDER BY justice_gap DESC
    LIMIT 15;
"""

result4 = pd.read_sql_query(q4, conn)
display(result4)

# =============================================================
# QUERY 5 — Data Coverage Check (Districts per State)
# =============================================================
# WHY: Validates our dataset is representative.
#      States with very few districts may skew state-level
#      averages and need to be flagged in the dashboard.

print("\n📊 QUERY 5: Data Coverage — Districts per State")
print("-" * 60)

q5 = """
    SELECT
        State,
        COUNT(DISTINCT District) AS districts_covered,
        COUNT(*)                 AS total_records,
        COUNT(DISTINCT Year)     AS years_covered
    FROM crime_data
    GROUP BY State
    ORDER BY districts_covered DESC;
"""

result5 = pd.read_sql_query(q5, conn)
display(result5)

# =============================================================
# QUERY 6 — National Yearly Trend (2021 → 2024)
# =============================================================
# WHY: Answers the macro question — Is India getting
#      safer or more dangerous for women over time?
#      This becomes the headline KPI in your Power BI dashboard.

print("\n📊 QUERY 6: National Yearly Crime Trend (2021-2024)")
print("-" * 60)

q6 = """
    SELECT
        Year,
        COUNT(DISTINCT District)          AS districts_reporting,
        SUM(Total_Violence)               AS national_total_crimes,
        ROUND(AVG(Crime_Rate), 2)         AS national_avg_crime_rate,
        ROUND(AVG(Police_Vacancy_Rate),2) AS national_avg_vacancy,
        ROUND(AVG(Sought_Help_Percent),2) AS national_avg_help_seeking
    FROM crime_data
    GROUP BY Year
    ORDER BY Year ASC;
"""

result6 = pd.read_sql_query(q6, conn)
display(result6)

# --- Close Connection ---
conn.close()
print("\n" + "=" * 60)
print("✅ STEP 3.2 COMPLETE — Exploratory Analysis Done")
print("   Next Step → Step 3.3: Critical Red Zone Query 🔒")
print("=" * 60)

In [ ]:
Step 3.3 — Critical Red Zone Identification Query
The Core Business Logic — Where Three Failures Collide

In [10]:
# =============================================================
# STEP 3.3 — Critical Red Zone Query (Exploratory Prototype)
# NOTE: This uses equal-weight NTILE scoring (3-12 scale)
# The production Risk Score is in Step 3.6 (0-100 weighted)
# This cell is kept for analytical exploration and SQL demo
# =============================================================

import pandas as pd
import sqlite3
import os

BASE_DIR = r"C:\Users\diyas\OneDrive\Desktop\women_safety_india"
DB_PATH  = os.path.join(BASE_DIR, "data", "cleaned", "women_safety.db")

conn = sqlite3.connect(DB_PATH)
print("✅ Connected to women_safety.db")
print("=" * 60)

# =============================================================
# STAGE 1 — COMPUTE NTILE SCORES FOR EACH DIMENSION
# =============================================================

print("\n📊 STAGE 1: Computing Risk Scores using NTILE Window Function")
print("-" * 60)

q_redzone = """
    WITH scored_districts AS (
        SELECT
            Year,
            State,
            District,
            Pop_2024_Est,
            Total_Violence,
            ROUND(Crime_Rate, 2)                AS Crime_Rate,
            ROUND(Police_Vacancy_Rate, 2)        AS Police_Vacancy_Rate,
            ROUND(Spousal_Violence_Percent, 2)   AS Spousal_Violence_Percent,
            ROUND(Sought_Help_Percent, 2)        AS Sought_Help_Percent,

            -- DIMENSION 1: Crime Rate Score
            -- Higher crime rate = higher score = more dangerous
            NTILE(4) OVER (
                PARTITION BY Year
                ORDER BY Crime_Rate ASC
            ) AS crime_score,

            -- DIMENSION 2: Police Vacancy Score
            -- Higher vacancy = higher score = less protected
            NTILE(4) OVER (
                PARTITION BY Year
                ORDER BY Police_Vacancy_Rate ASC
            ) AS vacancy_score,

            -- DIMENSION 3: Underreporting Score
            -- LOWER help seeking = HIGHER score = more underreporting
            -- We flip the order using DESC so low help = score 4
            NTILE(4) OVER (
                PARTITION BY Year
                ORDER BY Sought_Help_Percent DESC
            ) AS underreporting_score

        FROM crime_data
    ),

    -- STAGE 2: COMBINE SCORES INTO COMPOSITE RISK SCORE
    risk_scored AS (
        SELECT
            *,
            -- Composite score ranges from 3 (safest) to 12 (most dangerous)
            (crime_score + vacancy_score + underreporting_score) AS composite_risk_score,

            -- Classification based on composite score
            CASE
                WHEN (crime_score + vacancy_score + underreporting_score) = 12
                THEN 'CRITICAL RED ZONE'
                WHEN (crime_score + vacancy_score + underreporting_score) >= 10
                THEN 'HIGH RISK'
                WHEN (crime_score + vacancy_score + underreporting_score) >= 7
                THEN 'MODERATE RISK'
                ELSE 'LOWER RISK'
            END AS risk_classification,

            -- Rank within each year by composite score
            RANK() OVER (
                PARTITION BY Year
                ORDER BY
                    (crime_score + vacancy_score + underreporting_score) DESC
            ) AS national_risk_rank

        FROM scored_districts
    )

    -- STAGE 3: FINAL OUTPUT
    SELECT
        Year,
        State,
        District,
        Pop_2024_Est,
        Crime_Rate,
        Police_Vacancy_Rate,
        Spousal_Violence_Percent,
        Sought_Help_Percent,
        crime_score,
        vacancy_score,
        underreporting_score,
        composite_risk_score,
        risk_classification,
        national_risk_rank
    FROM risk_scored
    ORDER BY Year ASC, composite_risk_score DESC, national_risk_rank ASC;
"""

result_redzone = pd.read_sql_query(q_redzone, conn)
print(f"Total districts classified: {len(result_redzone)}")
display(result_redzone.head(30))

# =============================================================
# ISOLATE CRITICAL RED ZONES FOR DISPLAY
# =============================================================

print("\n📊 CRITICAL RED ZONES ONLY (Exploratory View):")
print("-" * 60)

critical_only = result_redzone[
    result_redzone['risk_classification'] == 'CRITICAL RED ZONE'
].copy()

print(f"Total Critical Red Zone records : {len(critical_only)}")
print(f"Unique Districts flagged        : {critical_only['District'].nunique()}")
print(f"States affected                 : {critical_only['State'].nunique()}")
display(critical_only)

# =============================================================
# SUMMARY BY RISK CLASSIFICATION
# =============================================================

print("\n📊 RISK CLASSIFICATION SUMMARY (Exploratory):")
print("-" * 60)

summary = result_redzone.groupby('risk_classification').agg(
    district_count   = ('District', 'count'),
    avg_crime_rate   = ('Crime_Rate', 'mean'),
    avg_vacancy_rate = ('Police_Vacancy_Rate', 'mean'),
    avg_help_seeking = ('Sought_Help_Percent', 'mean')
).round(2).reset_index()

display(summary)

conn.close()

print("\n" + "=" * 60)
print("✅ STEP 3.3 COMPLETE — Exploratory NTILE Analysis Done")
print("   NOTE: Production Risk Score is in Step 3.6")
print("   This step demonstrates SQL Window Function logic only")
print("=" * 60)

✅ Connected to women_safety.db

📊 STAGE 1: Computing Risk Scores using NTILE Window Function
------------------------------------------------------------
Total districts classified: 3976


,Year,State,District,Pop_2024_Est,Crime_Rate,Police_Vacancy_Rate,Spousal_Violence_Percent,Sought_Help_Percent,crime_score,vacancy_score,underreporting_score,composite_risk_score,risk_classification,national_risk_rank
0,2021,Uttar Pradesh,Khiri,3086497,12.02,0.29,34.8,2.4,4,4,4,12,CRITICAL RED ZONE,1
1,2021,Uttar Pradesh,Badaun,3086497,12.54,0.29,34.8,2.4,4,4,4,12,CRITICAL RED ZONE,1
2,2021,Uttar Pradesh,Pilibhit,2335658,13.57,0.29,34.8,2.4,4,4,4,12,CRITICAL RED ZONE,1
3,2021,Uttar Pradesh,Jhansi,2298393,13.57,0.29,34.8,2.4,4,4,4,12,CRITICAL RED ZONE,1
4,2021,Uttar Pradesh,Varanasi Commissionarate,3086497,14.16,0.29,34.8,2.4,4,4,4,12,CRITICAL RED ZONE,1
5,2021,Uttar Pradesh,Moradabad,5487806,14.40,0.29,34.8,2.4,4,4,4,12,CRITICAL RED ZONE,1
6,2021,Uttar Pradesh,Agra,5081616,15.61,0.29,34.8,2.4,4,4,4,12,CRITICAL RED ZONE,1
7,2021,Uttar Pradesh,Auraiya,1586476,16.89,0.29,34.8,2.4,4,4,4,12,CRITICAL RED ZONE,1
8,2021,Uttar Pradesh,Firozabad,2872879,17.09,0.29,34.8,2.4,4,4,4,12,CRITICAL RED ZONE,1
9,2021,Uttar Pradesh,Kanpur Commissionarate,3086497,17.30,0.29,34.8,2.4,4,4,4,12,CRITICAL RED ZONE,1



📊 CRITICAL RED ZONES ONLY (Exploratory View):
------------------------------------------------------------
Total Critical Red Zone records : 98
Unique Districts flagged        : 33
States affected                 : 3


,Year,State,District,Pop_2024_Est,Crime_Rate,Police_Vacancy_Rate,Spousal_Violence_Percent,Sought_Help_Percent,crime_score,vacancy_score,underreporting_score,composite_risk_score,risk_classification,national_risk_rank
0,2021,Uttar Pradesh,Khiri,3086497,12.02,0.29,34.8,2.4,4,4,4,12,CRITICAL RED ZONE,1
1,2021,Uttar Pradesh,Badaun,3086497,12.54,0.29,34.8,2.4,4,4,4,12,CRITICAL RED ZONE,1
2,2021,Uttar Pradesh,Pilibhit,2335658,13.57,0.29,34.8,2.4,4,4,4,12,CRITICAL RED ZONE,1
3,2021,Uttar Pradesh,Jhansi,2298393,13.57,0.29,34.8,2.4,4,4,4,12,CRITICAL RED ZONE,1
4,2021,Uttar Pradesh,Varanasi Commissionarate,3086497,14.16,0.29,34.8,2.4,4,4,4,12,CRITICAL RED ZONE,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2969,2024,Uttar Pradesh,Hamirpur,522983,11.28,0.29,34.8,2.4,4,4,4,12,CRITICAL RED ZONE,1
2970,2024,Jharkhand,Simdega,689514,13.34,0.23,31.5,2.9,4,4,4,12,CRITICAL RED ZONE,1
2971,2024,Gujarat,Total Districts,2894401,47.75,0.21,17.0,1.5,4,4,4,12,CRITICAL RED ZONE,1
2972,2024,Jharkhand,Total Districts,1648218,133.54,0.23,31.5,2.9,4,4,4,12,CRITICAL RED ZONE,1



📊 RISK CLASSIFICATION SUMMARY (Exploratory):
------------------------------------------------------------


,risk_classification,district_count,avg_crime_rate,avg_vacancy_rate,avg_help_seeking
0,CRITICAL RED ZONE,98,52.62,0.28,2.36
1,HIGH RISK,561,18.13,0.22,2.51
2,LOWER RISK,1219,3.52,0.11,5.67
3,MODERATE RISK,2098,18.44,0.17,4.41



✅ STEP 3.3 COMPLETE — Exploratory NTILE Analysis Done
   NOTE: Production Risk Score is in Step 3.6
   This step demonstrates SQL Window Function logic only


Step 3.4 — Year-over-Year Crime Trend Analysis
Using LAG Window Function to measure change over time

In [4]:
# =============================================================
# STEP 3.4 — Year-over-Year Trend Analysis using LAG
# =============================================================

import pandas as pd
import sqlite3
import os

BASE_DIR = r"C:\Users\diyas\OneDrive\Desktop\women_safety_india"
DB_PATH  = os.path.join(BASE_DIR, "data", "cleaned", "women_safety.db")

conn = sqlite3.connect(DB_PATH)
print("✅ Connected to women_safety.db")
print("=" * 60)

# =============================================================
# QUERY 1 — District Level YoY Crime Rate Change
# =============================================================

print("\n📊 QUERY 1: District Level Year-over-Year Crime Rate Change")
print("-" * 60)

q1 = """
    WITH district_yearly AS (
        SELECT
            Year,
            State,
            District,
            ROUND(Crime_Rate, 2)           AS Crime_Rate,
            ROUND(Police_Vacancy_Rate, 2)  AS Police_Vacancy_Rate,
            ROUND(Sought_Help_Percent, 2)  AS Sought_Help_Percent,

            -- LAG pulls the Crime_Rate value from the previous year
            -- for the same district in the same state
            LAG(Crime_Rate) OVER (
                PARTITION BY State, District
                ORDER BY Year ASC
            ) AS prev_year_crime_rate,

            LAG(Year) OVER (
                PARTITION BY State, District
                ORDER BY Year ASC
            ) AS prev_year

        FROM crime_data
    )

    SELECT
        Year,
        prev_year,
        State,
        District,
        Crime_Rate                         AS current_crime_rate,
        ROUND(prev_year_crime_rate, 2)     AS prev_crime_rate,

        -- Absolute change in crime rate
        ROUND(Crime_Rate - prev_year_crime_rate, 2) AS yoy_absolute_change,

        -- Percentage change in crime rate
        ROUND(
            ((Crime_Rate - prev_year_crime_rate) / prev_year_crime_rate) * 100
        , 2)                               AS yoy_percent_change,

        -- Flag whether situation improved or worsened
        CASE
            WHEN (Crime_Rate - prev_year_crime_rate) > 5
            THEN 'WORSENING'
            WHEN (Crime_Rate - prev_year_crime_rate) < -5
            THEN 'IMPROVING'
            ELSE 'STABLE'
        END                                AS trend_direction

    FROM district_yearly
    WHERE prev_year_crime_rate IS NOT NULL
    ORDER BY yoy_absolute_change DESC;
"""

result1 = pd.read_sql_query(q1, conn)
print(f"Total YoY records computed: {len(result1)}")
display(result1.head(20))

# =============================================================
# QUERY 2 — Most Rapidly Worsening Districts
# =============================================================

print("\n📊 QUERY 2: Top 15 Most Rapidly Worsening Districts")
print("-" * 60)

worsening = result1[result1['trend_direction'] == 'WORSENING'].copy()
worsening = worsening.sort_values('yoy_percent_change', ascending=False).head(15)
print(f"Total worsening districts: {len(result1[result1['trend_direction'] == 'WORSENING'])}")
display(worsening)

# =============================================================
# QUERY 3 — Most Rapidly Improving Districts
# =============================================================

print("\n📊 QUERY 3: Top 15 Most Rapidly Improving Districts")
print("-" * 60)

improving = result1[result1['trend_direction'] == 'IMPROVING'].copy()
improving = improving.sort_values('yoy_percent_change', ascending=True).head(15)
print(f"Total improving districts: {len(result1[result1['trend_direction'] == 'IMPROVING'])}")
display(improving)

# =============================================================
# QUERY 4 — State Level YoY Trend Summary
# =============================================================

print("\n📊 QUERY 4: State Level YoY Trend Summary")
print("-" * 60)

q4 = """
    WITH district_yearly AS (
        SELECT
            Year,
            State,
            District,
            Crime_Rate,

            LAG(Crime_Rate) OVER (
                PARTITION BY State, District
                ORDER BY Year ASC
            ) AS prev_year_crime_rate

        FROM crime_data
    ),

    changes AS (
        SELECT
            Year,
            State,
            District,
            Crime_Rate,
            prev_year_crime_rate,
            (Crime_Rate - prev_year_crime_rate) AS yoy_change
        FROM district_yearly
        WHERE prev_year_crime_rate IS NOT NULL
    )

    SELECT
        State,
        COUNT(DISTINCT District)          AS districts_tracked,
        ROUND(AVG(yoy_change), 2)         AS avg_yoy_change,
        ROUND(MAX(yoy_change), 2)         AS worst_district_change,
        ROUND(MIN(yoy_change), 2)         AS best_district_change,
        SUM(CASE WHEN yoy_change > 5
            THEN 1 ELSE 0 END)            AS worsening_districts,
        SUM(CASE WHEN yoy_change < -5
            THEN 1 ELSE 0 END)            AS improving_districts
    FROM changes
    GROUP BY State
    ORDER BY avg_yoy_change DESC;
"""

result4 = pd.read_sql_query(q4, conn)
display(result4)

# =============================================================
# QUERY 5 — National Trend Direction Summary
# =============================================================

print("\n📊 QUERY 5: National Trend Direction Summary")
print("-" * 60)

trend_summary = result1.groupby(['Year', 'trend_direction']).agg(
    district_count = ('District', 'count')
).reset_index()

display(trend_summary)

# =============================================================
# SAVE YoY RESULTS FOR POWER BI
# =============================================================

OUTPUT_PATH = os.path.join(BASE_DIR, "data", "final", "yoy_trend_analysis.csv")
result1.to_csv(OUTPUT_PATH, index=False)
print(f"\n✅ YoY Trend data saved to: {OUTPUT_PATH}")

STATE_PATH = os.path.join(BASE_DIR, "data", "final", "state_yoy_summary.csv")
result4.to_csv(STATE_PATH, index=False)
print(f"✅ State YoY Summary saved to: {STATE_PATH}")

conn.close()

print("\n" + "=" * 60)
print("✅ STEP 3.4 COMPLETE — YoY Trend Analysis Done")
print("   Next Step → Step 3.5: Underreporting & Justice Gap 🔒")
print("=" * 60)

✅ Connected to women_safety.db

📊 QUERY 1: District Level Year-over-Year Crime Rate Change
------------------------------------------------------------
Total YoY records computed: 2915


,Year,prev_year,State,District,current_crime_rate,prev_crime_rate,yoy_absolute_change,yoy_percent_change,trend_direction
0,2022,2022,Delhi,North,670.45,33.00,637.45,1931.59,WORSENING
1,2021,2021,Delhi,North,604.79,29.77,575.02,1931.57,WORSENING
2,2023,2023,Delhi,North,499.35,24.58,474.77,1931.57,WORSENING
3,2022,2022,Delhi,South,377.20,20.28,356.92,1760.37,WORSENING
4,2021,2021,Delhi,West,356.28,19.11,337.17,1764.08,WORSENING
5,2022,2022,Delhi,West,302.10,16.21,285.89,1764.05,WORSENING
6,2021,2021,Delhi,South,277.72,14.93,262.79,1760.38,WORSENING
7,2024,2024,Delhi,North,260.62,12.83,247.79,1931.59,WORSENING
8,2023,2023,Delhi,South,257.58,13.85,243.73,1760.33,WORSENING
9,2023,2023,Delhi,West,257.49,13.81,243.68,1764.08,WORSENING



📊 QUERY 2: Top 15 Most Rapidly Worsening Districts
------------------------------------------------------------
Total worsening districts: 135


,Year,prev_year,State,District,current_crime_rate,prev_crime_rate,yoy_absolute_change,yoy_percent_change,trend_direction
30,2024,2024,Tripura,North,43.77,2.15,41.62,1931.67,WORSENING
12,2023,2023,Tripura,North,113.40,5.58,107.82,1931.60,WORSENING
7,2024,2024,Delhi,North,260.62,12.83,247.79,1931.59,WORSENING
14,2022,2022,Tripura,North,111.41,5.48,105.93,1931.59,WORSENING
0,2022,2022,Delhi,North,670.45,33.00,637.45,1931.59,WORSENING
20,2021,2021,Tripura,North,73.61,3.62,69.99,1931.58,WORSENING
2,2023,2023,Delhi,North,499.35,24.58,474.77,1931.57,WORSENING
1,2021,2021,Delhi,North,604.79,29.77,575.02,1931.57,WORSENING
22,2021,2021,Tripura,West,68.20,3.66,64.54,1764.17,WORSENING
4,2021,2021,Delhi,West,356.28,19.11,337.17,1764.08,WORSENING



📊 QUERY 3: Top 15 Most Rapidly Improving Districts
------------------------------------------------------------
Total improving districts: 463


...


📊 QUERY 4: State Level YoY Trend Summary
------------------------------------------------------------


,State,districts_tracked,avg_yoy_change,worst_district_change,best_district_change,worsening_districts,improving_districts
0,Goa,4,2.53,14.07,-0.41,3,0
1,Nagaland,18,0.62,7.25,-1.72,1,0
2,Himachal Pradesh,18,0.46,13.86,-6.83,3,1
3,Lakshadweep,2,0.45,5.39,-4.05,2,0
4,D&N Haveli and Daman & Diu,4,0.45,7.15,-10.72,2,2
5,Tripura,10,0.35,107.82,-111.24,12,10
6,Puducherry,3,0.30,1.21,-0.87,0,0
7,Mizoram,15,0.26,10.14,-11.41,2,1
8,Sikkim,12,0.22,2.05,-0.33,0,0
9,Meghalaya,14,0.20,11.09,-7.39,1,2



📊 QUERY 5: National Trend Direction Summary
------------------------------------------------------------


,Year,trend_direction,district_count
0,2021,IMPROVING,4
1,2021,STABLE,9
2,2021,WORSENING,7
3,2022,IMPROVING,63
4,2022,STABLE,792
5,2022,WORSENING,71
6,2023,IMPROVING,86
7,2023,STABLE,848
8,2023,WORSENING,28
9,2024,IMPROVING,310



✅ YoY Trend data saved to: C:\Users\diyas\OneDrive\Desktop\women_safety_india\data\final\yoy_trend_analysis.csv
✅ State YoY Summary saved to: C:\Users\diyas\OneDrive\Desktop\women_safety_india\data\final\state_yoy_summary.csv

✅ STEP 3.4 COMPLETE — YoY Trend Analysis Done
   Next Step → Step 3.5: Underreporting & Justice Gap 🔒


Step 3.5 — Underreporting & Justice Gap Deep Dive
Final SQL Step — Quantifying Systemic Failure Zones

In [7]:
# =============================================================
# STEP 3.5 — Underreporting & Justice Gap Deep Dive
# =============================================================

import pandas as pd
import sqlite3
import os

BASE_DIR = r"C:\Users\diyas\OneDrive\Desktop\women_safety_india"
DB_PATH  = os.path.join(BASE_DIR, "data", "cleaned", "women_safety.db")

conn = sqlite3.connect(DB_PATH)
print("✅ Connected to women_safety.db")
print("=" * 60)

# =============================================================
# QUERY 1 — Justice Gap Calculation at District Level
# =============================================================

print("\n📊 QUERY 1: Justice Gap — District Level")
print("-" * 60)

q1 = """
    SELECT
        Year,
        State,
        District,
        ROUND(Spousal_Violence_Percent, 2)                        AS spousal_violence_pct,
        ROUND(Sought_Help_Percent, 2)                             AS sought_help_pct,
        ROUND(Spousal_Violence_Percent - Sought_Help_Percent, 2)  AS justice_gap,

        CASE
            WHEN (Spousal_Violence_Percent - Sought_Help_Percent) >= 40
            THEN 'CRITICAL GAP'
            WHEN (Spousal_Violence_Percent - Sought_Help_Percent) >= 25
            THEN 'HIGH GAP'
            WHEN (Spousal_Violence_Percent - Sought_Help_Percent) >= 10
            THEN 'MODERATE GAP'
            ELSE 'LOW GAP'
        END                                                        AS gap_severity,

        ROUND(Crime_Rate, 2)          AS crime_rate,
        ROUND(Police_Vacancy_Rate, 2) AS police_vacancy_rate

    FROM crime_data
    ORDER BY justice_gap DESC;
"""

result1 = pd.read_sql_query(q1, conn)
print(f"Total records: {len(result1)}")
display(result1.head(20))

# =============================================================
# QUERY 2 — Justice Gap Severity Distribution
# =============================================================

print("\n📊 QUERY 2: Justice Gap Severity Distribution")
print("-" * 60)

q2 = """
    SELECT
        CASE
            WHEN (Spousal_Violence_Percent - Sought_Help_Percent) >= 40
            THEN 'CRITICAL GAP'
            WHEN (Spousal_Violence_Percent - Sought_Help_Percent) >= 25
            THEN 'HIGH GAP'
            WHEN (Spousal_Violence_Percent - Sought_Help_Percent) >= 10
            THEN 'MODERATE GAP'
            ELSE 'LOW GAP'
        END                                AS gap_severity,

        COUNT(*)                           AS district_count,
        ROUND(AVG(Crime_Rate), 2)          AS avg_crime_rate,
        ROUND(AVG(Police_Vacancy_Rate), 2) AS avg_police_vacancy,
        ROUND(AVG(Spousal_Violence_Percent), 2) AS avg_spousal_violence,
        ROUND(AVG(Sought_Help_Percent), 2) AS avg_help_seeking

    FROM crime_data
    GROUP BY gap_severity
    ORDER BY district_count DESC;
"""

result2 = pd.read_sql_query(q2, conn)
display(result2)

# =============================================================
# QUERY 3 — Triple Failure Zones
  #Districts where High Crime + High Justice Gap + High Vacancy
  #all exist simultaneously
  #This is the most extreme form of systemic failure
# =============================================================

print("\n📊 QUERY 3: Triple Failure Zones")
print("-" * 60)

q3 = """
    SELECT
        Year,
        State,
        District,
        ROUND(Crime_Rate, 2)                                      AS crime_rate,
        ROUND(Police_Vacancy_Rate, 2)                             AS police_vacancy_rate,
        ROUND(Spousal_Violence_Percent, 2)                        AS spousal_violence_pct,
        ROUND(Sought_Help_Percent, 2)                             AS sought_help_pct,
        ROUND(Spousal_Violence_Percent - Sought_Help_Percent, 2)  AS justice_gap,
        Pop_2024_Est

    FROM crime_data

    WHERE Crime_Rate > (
            SELECT AVG(Crime_Rate) + (0.5 * AVG(Crime_Rate))
            FROM crime_data
          )
    AND Police_Vacancy_Rate > (
            SELECT AVG(Police_Vacancy_Rate)
            FROM crime_data
          )
    AND (Spousal_Violence_Percent - Sought_Help_Percent) > (
            SELECT AVG(Spousal_Violence_Percent - Sought_Help_Percent)
            FROM crime_data
          )

    ORDER BY justice_gap DESC, crime_rate DESC;
"""

result3 = pd.read_sql_query(q3, conn)
print(f"Total Triple Failure Zone districts: {len(result3)}")
display(result3.head(20))

# =============================================================
# QUERY 4 — State Level Justice Gap Summary
# =============================================================

print("\n📊 QUERY 4: State Level Justice Gap Summary")
print("-" * 60)

q4 = """
    SELECT
        State,
        COUNT(DISTINCT District)                                        AS districts,
        ROUND(AVG(Spousal_Violence_Percent), 2)                        AS avg_spousal_violence,
        ROUND(AVG(Sought_Help_Percent), 2)                             AS avg_help_seeking,
        ROUND(AVG(Spousal_Violence_Percent - Sought_Help_Percent), 2)  AS avg_justice_gap,
        ROUND(MAX(Spousal_Violence_Percent - Sought_Help_Percent), 2)  AS worst_justice_gap,
        SUM(CASE
                WHEN (Spousal_Violence_Percent - Sought_Help_Percent) >= 40
                THEN 1 ELSE 0
            END)                                                        AS critical_gap_districts

    FROM crime_data
    GROUP BY State
    ORDER BY avg_justice_gap DESC;
"""

result4 = pd.read_sql_query(q4, conn)
display(result4)

# =============================================================
# QUERY 5 — Population at Risk in Critical Gap Zones
# =============================================================

print("\n📊 QUERY 5: Population at Risk in Critical Gap Zones")
print("-" * 60)

q5 = """
    SELECT
        SUM(CASE
                WHEN (Spousal_Violence_Percent - Sought_Help_Percent) >= 40
                THEN Pop_2024_Est ELSE 0
            END)                AS population_in_critical_gap_zones,

        SUM(CASE
                WHEN (Spousal_Violence_Percent - Sought_Help_Percent) >= 25
                THEN Pop_2024_Est ELSE 0
            END)                AS population_in_high_gap_zones,

        SUM(Pop_2024_Est)       AS total_population_covered,

        ROUND(
            100.0 * SUM(CASE
                WHEN (Spousal_Violence_Percent - Sought_Help_Percent) >= 40
                THEN Pop_2024_Est ELSE 0
            END) / SUM(Pop_2024_Est)
        , 2)                    AS pct_population_in_critical_zones

    FROM crime_data
    WHERE Year = 2024;
"""

result5 = pd.read_sql_query(q5, conn)
display(result5)

# =============================================================
# SAVE ALL OUTPUTS FOR POWER BI
# =============================================================

path1 = os.path.join(BASE_DIR, "data", "final", "justice_gap_district.csv")
result1.to_csv(path1, index=False)
print(f"\n✅ Justice Gap District data saved to: {path1}")

path2 = os.path.join(BASE_DIR, "data", "final", "triple_failure_zones.csv")
result3.to_csv(path2, index=False)
print(f"✅ Triple Failure Zones saved to: {path2}")

path3 = os.path.join(BASE_DIR, "data", "final", "state_justice_gap.csv")
result4.to_csv(path3, index=False)
print(f"✅ State Justice Gap Summary saved to: {path3}")

conn.close()

print("\n" + "=" * 60)
print("✅ STEP 3.5 COMPLETE — All SQL Analysis Done")
print("   SQL Layer is 100% Complete ✅")
print("   Next Step → Step 4: Power BI Dashboard 🔒")
print("=" * 60)

✅ Connected to women_safety.db

📊 QUERY 1: Justice Gap — District Level
------------------------------------------------------------
Total records: 3976


,Year,State,District,spousal_violence_pct,sought_help_pct,justice_gap,gap_severity,crime_rate,police_vacancy_rate
0,2021,Karnataka,Bagalkot,44.4,3.8,40.6,CRITICAL GAP,2.07,0.18
1,2021,Karnataka,Bengaluru City,44.4,3.8,40.6,CRITICAL GAP,32.21,0.18
2,2021,Karnataka,Bengaluru District,44.4,3.8,40.6,CRITICAL GAP,6.83,0.18
3,2021,Karnataka,Belagavi District,44.4,3.8,40.6,CRITICAL GAP,8.29,0.18
4,2021,Karnataka,Ballari,44.4,3.8,40.6,CRITICAL GAP,6.61,0.18
5,2021,Karnataka,Bidar,44.4,3.8,40.6,CRITICAL GAP,3.88,0.18
6,2021,Karnataka,Vijayapura,44.4,3.8,40.6,CRITICAL GAP,6.39,0.18
7,2021,Karnataka,Chikkaballapura,44.4,3.8,40.6,CRITICAL GAP,5.06,0.18
8,2021,Karnataka,Chamarajnagar,44.4,3.8,40.6,CRITICAL GAP,1.68,0.18
9,2021,Karnataka,Chikkamagaluru,44.4,3.8,40.6,CRITICAL GAP,4.71,0.18



📊 QUERY 2: Justice Gap Severity Distribution
------------------------------------------------------------


,gap_severity,district_count,avg_crime_rate,avg_police_vacancy,avg_spousal_violence,avg_help_seeking
0,MODERATE GAP,1618,16.35,0.16,22.85,4.06
1,HIGH GAP,1451,12.63,0.21,34.98,4.40
2,LOW GAP,752,16.19,0.06,15.50,6.03
3,CRITICAL GAP,155,8.68,0.18,44.40,3.80



📊 QUERY 3: Triple Failure Zones
------------------------------------------------------------
Total Triple Failure Zone districts: 75


...


📊 QUERY 4: State Level Justice Gap Summary
------------------------------------------------------------


,State,districts,avg_spousal_violence,avg_help_seeking,avg_justice_gap,worst_justice_gap,critical_gap_districts
0,Karnataka,41,44.40,3.8,40.6,40.6,155
1,Bihar,47,40.00,3.5,36.5,36.5,0
2,Manipur,19,39.60,4.4,35.2,35.2,0
3,Tamil Nadu,51,38.10,5.3,32.8,32.8,0
4,Uttar Pradesh,80,34.80,2.4,32.4,32.4,0
5,Telangana,31,36.90,7.6,29.3,29.3,0
6,Jharkhand,27,31.50,2.9,28.6,28.6,0
7,Assam,43,32.00,5.9,26.1,26.1,0
8,Andhra Pradesh,39,30.00,4.4,25.6,25.6,0
9,Odisha,38,30.40,5.4,25.0,25.0,0



📊 QUERY 5: Population at Risk in Critical Gap Zones
------------------------------------------------------------


,population_in_critical_gap_zones,population_in_high_gap_zones,total_population_covered,pct_population_in_critical_zones
0,69620817,1009956741,2143938633,3.25



✅ Justice Gap District data saved to: C:\Users\diyas\OneDrive\Desktop\women_safety_india\data\final\justice_gap_district.csv
✅ Triple Failure Zones saved to: C:\Users\diyas\OneDrive\Desktop\women_safety_india\data\final\triple_failure_zones.csv
✅ State Justice Gap Summary saved to: C:\Users\diyas\OneDrive\Desktop\women_safety_india\data\final\state_justice_gap.csv

✅ STEP 3.5 COMPLETE — All SQL Analysis Done
   SQL Layer is 100% Complete ✅
   Next Step → Step 4: Power BI Dashboard 🔒


Step 3.6 — Weighted Normalized Risk Score (0-100)
Project Intellectual Property — The Core Scoring Engine

In [9]:
# =============================================================
# STEP 3.6 — Weighted Normalized Risk Score (0-100)
# FIXED VERSION — Full NaN and Inf handling at every stage
# =============================================================

import pandas as pd
import numpy as np
import sqlite3
import os

BASE_DIR = r"C:\Users\diyas\OneDrive\Desktop\women_safety_india"
DB_PATH  = os.path.join(BASE_DIR, "data", "cleaned", "women_safety.db")

conn = sqlite3.connect(DB_PATH)
print("✅ Connected to women_safety.db")
print("=" * 60)

# =============================================================
# STEP 1 — LOAD BASE DATA
# =============================================================

print("\n[1/8] Loading base data from database...")

df = pd.read_sql_query("SELECT * FROM crime_data;", conn)

print(f"      ✅ Loaded {len(df)} records")

# =============================================================
# STEP 2 — DIAGNOSE NULL VALUES BEFORE ANYTHING ELSE
# =============================================================

print("\n[2/8] Diagnosing NULL values in key columns...")

key_cols = [
    'Crime_Rate',
    'Police_Vacancy_Rate',
    'Spousal_Violence_Percent',
    'Sought_Help_Percent'
]

for col in key_cols:
    null_count = df[col].isnull().sum()
    inf_count  = np.isinf(df[col].replace([None], np.nan)
                  .astype(float)).sum()
    print(f"      {col:<30} NaN: {null_count}   Inf: {inf_count}")

# =============================================================
# STEP 3 — IMPUTE REMAINING NaNs WITH YEAR + STATE MEAN
# =============================================================
# If a district has NaN in any scoring column we fill it
# with the mean of all other districts in the same
# State and Year. This is the same strategy used in
# Step 2 data cleaning and is the most defensible
# imputation for geographic panel data.

print("\n[3/8] Imputing remaining NaNs with Year+State mean...")

for col in key_cols:
    before = df[col].isnull().sum()
    
    # First try Year + State mean
    df[col] = df.groupby(['Year', 'State'])[col].transform(
        lambda x: x.fillna(x.mean())
    )
    
    # If still NaN (entire state-year group was NaN)
    # fall back to Year mean
    df[col] = df.groupby(['Year'])[col].transform(
        lambda x: x.fillna(x.mean())
    )
    
    # Final fallback — global column mean
    df[col] = df[col].fillna(df[col].mean())
    
    after = df[col].isnull().sum()
    print(f"      {col:<30} Before: {before}  After: {after}")

print("      ✅ Imputation complete — zero NaNs remaining")

# =============================================================
# STEP 4 — ENGINEER JUSTICE GAP
# =============================================================

print("\n[4/8] Engineering Justice Gap metric...")

df['Justice_Gap'] = (
    df['Spousal_Violence_Percent'] - df['Sought_Help_Percent']
)

# Clip negatives to 0 — negative gap is a data artifact
df['Justice_Gap'] = df['Justice_Gap'].clip(lower=0)

# Replace any inf values just in case
df['Justice_Gap'] = df['Justice_Gap'].replace(
    [np.inf, -np.inf], 0
)

print(f"      ✅ Justice_Gap created")
print(f"      Min  : {df['Justice_Gap'].min():.2f}")
print(f"      Max  : {df['Justice_Gap'].max():.2f}")
print(f"      Mean : {df['Justice_Gap'].mean():.2f}")
print(f"      NaNs : {df['Justice_Gap'].isnull().sum()}")

# =============================================================
# STEP 5 — MIN MAX NORMALIZATION PER YEAR
# =============================================================

print("\n[5/8] Applying Min-Max Normalization per Year...")

def min_max_normalize(series):
    min_val = series.min()
    max_val = series.max()
    # Edge case: all values identical in this group
    if max_val == min_val:
        return pd.Series([0.5] * len(series), index=series.index)
    normalized = (series - min_val) / (max_val - min_val)
    # Clip to 0-1 strictly to handle any floating point drift
    return normalized.clip(0, 1)

df['Crime_Rate_Norm']   = df.groupby('Year')['Crime_Rate'].transform(
    min_max_normalize
)
df['Vacancy_Rate_Norm'] = df.groupby('Year')['Police_Vacancy_Rate'].transform(
    min_max_normalize
)
df['Justice_Gap_Norm']  = df.groupby('Year')['Justice_Gap'].transform(
    min_max_normalize
)

# Verify no NaNs crept in after normalization
for col in ['Crime_Rate_Norm', 'Vacancy_Rate_Norm', 'Justice_Gap_Norm']:
    nan_check = df[col].isnull().sum()
    print(f"      {col:<25} NaN after normalization: {nan_check}")

print("      ✅ Normalization complete")

# =============================================================
# STEP 6 — COMPUTE WEIGHTED RISK SCORE
# =============================================================

print("\n[6/8] Computing Weighted Risk Score (0-100)...")

WEIGHT_CRIME   = 0.40
WEIGHT_VACANCY = 0.30
WEIGHT_JUSTICE = 0.30

assert round(WEIGHT_CRIME + WEIGHT_VACANCY + WEIGHT_JUSTICE, 10) == 1.0, \
    "Weights must sum to 1.0"

df['Risk_Score'] = (
    (df['Crime_Rate_Norm']   * WEIGHT_CRIME)   +
    (df['Vacancy_Rate_Norm'] * WEIGHT_VACANCY) +
    (df['Justice_Gap_Norm']  * WEIGHT_JUSTICE)
) * 100

# Replace any inf values
df['Risk_Score'] = df['Risk_Score'].replace(
    [np.inf, -np.inf], np.nan
)

# Final NaN check on Risk Score before proceeding
risk_nan = df['Risk_Score'].isnull().sum()
print(f"      NaN in Risk_Score before fix: {risk_nan}")

if risk_nan > 0:
    # Fill remaining NaN risk scores with year mean
    df['Risk_Score'] = df.groupby('Year')['Risk_Score'].transform(
        lambda x: x.fillna(x.mean())
    )
    print(f"      NaN in Risk_Score after fix : {df['Risk_Score'].isnull().sum()}")

df['Risk_Score'] = df['Risk_Score'].round(2)

print(f"      ✅ Risk Score computed")
print(f"      Min  : {df['Risk_Score'].min():.2f}")
print(f"      Max  : {df['Risk_Score'].max():.2f}")
print(f"      Mean : {df['Risk_Score'].mean():.2f}")
print(f"      Std  : {df['Risk_Score'].std():.2f}")
print(f"      NaNs : {df['Risk_Score'].isnull().sum()}")

# =============================================================
# STEP 7 — CLASSIFY RISK TIERS
# =============================================================

print("\n[7/8] Classifying Risk Tiers...")

def classify_risk(score):
    if pd.isnull(score):
        return 'UNCLASSIFIED'
    elif score >= 75:
        return 'CRITICAL RED ZONE'
    elif score >= 50:
        return 'HIGH RISK'
    elif score >= 25:
        return 'MODERATE RISK'
    else:
        return 'LOWER RISK'

df['Risk_Tier'] = df['Risk_Score'].apply(classify_risk)

print("\n      Risk Tier Distribution:")
print(df['Risk_Tier'].value_counts().to_string())

# =============================================================
# STEP 8 — NATIONAL RISK RANK (Safe Integer Conversion)
# =============================================================

print("\n[8/8] Computing National Risk Rank per Year...")

# Use float first then convert to int safely
# Only convert after confirming zero NaNs
df['National_Risk_Rank'] = df.groupby('Year')['Risk_Score'].rank(
    ascending = False,
    method    = 'dense',
    na_option = 'bottom'   # push any NaN ranks to the bottom
)

# Now safely convert to int
df['National_Risk_Rank'] = df['National_Risk_Rank'].fillna(
    df['National_Risk_Rank'].max()
).astype(int)

print("      ✅ National Risk Rank assigned safely")

# =============================================================
# BUILD FINAL OUTPUT TABLE
# =============================================================

final_df = df[[
    'Year',
    'State',
    'District',
    'Pop_2024_Est',
    'Total_Violence',
    'Crime_Rate',
    'Police_Vacancy_Rate',
    'Spousal_Violence_Percent',
    'Sought_Help_Percent',
    'Justice_Gap',
    'Crime_Rate_Norm',
    'Vacancy_Rate_Norm',
    'Justice_Gap_Norm',
    'Risk_Score',
    'Risk_Tier',
    'National_Risk_Rank'
]].copy()

final_df = final_df.sort_values(
    ['Year', 'Risk_Score'],
    ascending = [True, False]
).reset_index(drop=True)

print(f"\n      ✅ Final table shape: {final_df.shape}")

# =============================================================
# PREVIEWS
# =============================================================

print("\n📊 TOP 20 HIGHEST RISK DISTRICTS IN 2024:")
print("-" * 60)
top20 = final_df[final_df['Year'] == 2024].head(20)
display(top20[[
    'State', 'District', 'Risk_Score',
    'Risk_Tier', 'National_Risk_Rank',
    'Crime_Rate', 'Police_Vacancy_Rate', 'Justice_Gap'
]])

print("\n📊 CRITICAL RED ZONES SUMMARY:")
print("-" * 60)
critical = final_df[final_df['Risk_Tier'] == 'CRITICAL RED ZONE'].copy()
print(f"      Total Critical Records : {len(critical)}")
print(f"      Unique Districts       : {critical['District'].nunique()}")
print(f"      States Affected        : {critical['State'].nunique()}")
display(critical[[
    'Year', 'State', 'District',
    'Risk_Score', 'National_Risk_Rank',
    'Crime_Rate', 'Police_Vacancy_Rate', 'Justice_Gap'
]].head(30))

print("\n📊 YEARLY RISK SUMMARY:")
print("-" * 60)
yearly = final_df.groupby('Year').agg(
    avg_risk_score   = ('Risk_Score', 'mean'),
    max_risk_score   = ('Risk_Score', 'max'),
    min_risk_score   = ('Risk_Score', 'min'),
    critical_zones   = ('Risk_Tier', lambda x: (x == 'CRITICAL RED ZONE').sum()),
    high_risk        = ('Risk_Tier', lambda x: (x == 'HIGH RISK').sum()),
    moderate_risk    = ('Risk_Tier', lambda x: (x == 'MODERATE RISK').sum()),
    lower_risk       = ('Risk_Tier', lambda x: (x == 'LOWER RISK').sum())
).round(2).reset_index()

display(yearly)

# =============================================================
# SAVE TO DATABASE
# =============================================================

print("\n💾 Saving to SQLite database...")

final_df.to_sql(
    name      = 'risk_scores',
    con       = conn,
    if_exists = 'replace',
    index     = False
)

print("      ✅ Table 'risk_scores' saved to women_safety.db")

# Verification query
verify = pd.read_sql_query("""
    SELECT
        Year,
        COUNT(*)                  AS total_districts,
        ROUND(AVG(Risk_Score), 2) AS avg_risk_score,
        ROUND(MAX(Risk_Score), 2) AS max_risk_score,
        COUNT(CASE WHEN Risk_Tier = 'CRITICAL RED ZONE'
              THEN 1 END)         AS critical_zones
    FROM risk_scores
    GROUP BY Year
    ORDER BY Year;
""", conn)

print("\n📊 DATABASE VERIFICATION:")
display(verify)

# =============================================================
# SAVE TO CSV
# =============================================================

MASTER_PATH = os.path.join(
    BASE_DIR, "data", "final", "master_risk_scores.csv"
)

final_df.to_csv(MASTER_PATH, index=False)
print(f"\n✅ master_risk_scores.csv saved to data/final/")

conn.close()

print("\n" + "=" * 60)
print("✅ STEP 3.6 COMPLETE — Risk Score Engine Built")
print("   master_risk_scores.csv → Ready for Power BI")
print("=" * 60)

✅ Connected to women_safety.db

[1/8] Loading base data from database...
      ✅ Loaded 3976 records

[2/8] Diagnosing NULL values in key columns...
      Crime_Rate                     NaN: 0   Inf: 0
      Police_Vacancy_Rate            NaN: 0   Inf: 0
      Spousal_Violence_Percent       NaN: 0   Inf: 0
      Sought_Help_Percent            NaN: 228   Inf: 0

[3/8] Imputing remaining NaNs with Year+State mean...
      Crime_Rate                     Before: 0  After: 0
      Police_Vacancy_Rate            Before: 0  After: 0
      Spousal_Violence_Percent       Before: 0  After: 0
      Sought_Help_Percent            Before: 228  After: 0
      ✅ Imputation complete — zero NaNs remaining

[4/8] Engineering Justice Gap metric...
      ✅ Justice_Gap created
      Min  : 0.00
      Max  : 40.60
      Mean : 22.29
      NaNs : 0

[5/8] Applying Min-Max Normalization per Year...
      Crime_Rate_Norm           NaN after normalization: 0
      Vacancy_Rate_Norm         NaN after normalizati

,State,District,Risk_Score,Risk_Tier,National_Risk_Rank,Crime_Rate,Police_Vacancy_Rate,Justice_Gap
2961,Uttar Pradesh,Total Districts,69.20,HIGH RISK,1,208.099992,0.289073,32.4
2962,Rajasthan,Total Districts,67.45,HIGH RISK,2,429.786917,0.128299,21.6
2963,Madhya Pradesh,Total Districts,66.23,HIGH RISK,3,386.374640,0.142481,23.7
2964,Bihar,Total Districts,64.27,HIGH RISK,4,78.466635,0.334936,36.5
2965,Bihar,Purnea,57.50,HIGH RISK,5,5.677004,0.334936,36.5
2966,Bihar,Kishanganj,57.45,HIGH RISK,6,5.144141,0.334936,36.5
2967,Bihar,Patna,57.28,HIGH RISK,7,3.321302,0.334936,36.5
2968,Bihar,Gaya,57.27,HIGH RISK,8,3.247441,0.334936,36.5
2969,Bihar,Katihar,57.27,HIGH RISK,8,3.256238,0.334936,36.5
2970,Bihar,Araria,57.24,HIGH RISK,9,2.845387,0.334936,36.5



📊 CRITICAL RED ZONES SUMMARY:
------------------------------------------------------------
      Total Critical Records : 3
      Unique Districts       : 1
      States Affected        : 1


,Year,State,District,Risk_Score,National_Risk_Rank,Crime_Rate,Police_Vacancy_Rate,Justice_Gap
0,2021,Uttar Pradesh,Total Districts,76.56,1,752.762760,0.289073,32.4
964,2022,Uttar Pradesh,Total Districts,83.47,1,835.996277,0.289073,32.4
1951,2023,Uttar Pradesh,Total Districts,84.36,1,818.371118,0.289073,32.4



📊 YEARLY RISK SUMMARY:
------------------------------------------------------------


,Year,avg_risk_score,max_risk_score,min_risk_score,critical_zones,high_risk,moderate_risk,lower_risk
0,2021,31.66,76.56,7.27,1,115,597,251
1,2022,31.68,83.47,7.27,1,117,617,252
2,2023,31.52,84.36,7.27,1,117,628,264
3,2024,31.51,69.20,7.27,0,106,646,263



💾 Saving to SQLite database...
      ✅ Table 'risk_scores' saved to women_safety.db

📊 DATABASE VERIFICATION:


,Year,total_districts,avg_risk_score,max_risk_score,critical_zones
0,2021,964,31.66,76.56,1
1,2022,987,31.68,83.47,1
2,2023,1010,31.52,84.36,1
3,2024,1015,31.51,69.20,0



✅ master_risk_scores.csv saved to data/final/

✅ STEP 3.6 COMPLETE — Risk Score Engine Built
   master_risk_scores.csv → Ready for Power BI
